In [254]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn

In [255]:
class FFNN(nn.Module):

    def __init__(self, h_size: int, out_n: int, drop: float=None):
        super().__init__()
        self.l1 = nn.Linear(42,h_size)
        self.l2 = nn.Linear(h_size, h_size)
        self.l3 = nn.Linear(h_size, h_size // 2)
        self.l4 = nn.Linear(h_size // 2, out_n)
        if drop:
            self.d1 = nn.Dropout(drop)
            self.d2 = nn.Dropout(drop)
        else:
            self.d1 = self.d2 = None
    
    def forward(self, X):
        """
        Forward pass of the neural network
        """
        X = F.relu(self.l1(X))
        X = F.relu(self.l2(X))
        if self.d1:
            X = self.d1(X)
        X = F.relu(self.l3(X))
        if self.d2:
            X = self.d2(X)
        X = F.relu(self.l4(X))
        # return F.softmax(X)
        return X

In [256]:
device = 'cuda'

In [257]:
import os
os.getcwd()

'c:\\Users\\Lucas\\OneDrive\\A Level\\Computer Science Code\\Hand Detection NEA\\Real Thing\\repo'

In [258]:
data = pd.read_csv('c:\\Users\\Lucas\\OneDrive\\A Level\\Computer Science Code\\Hand Detection NEA\\Real Thing\\data.csv')
cols = data.columns
print(cols)
data = data.sample(frac=1)

Index(['label', 'hand', 'data0', 'data1', 'data2', 'data3', 'data4', 'data5',
       'data6', 'data7', 'data8', 'data9', 'data10', 'data11', 'data12',
       'data13', 'data14', 'data15', 'data16', 'data17', 'data18', 'data19',
       'data20', 'data21', 'data22', 'data23', 'data24', 'data25', 'data26',
       'data27', 'data28', 'data29', 'data30', 'data31', 'data32', 'data33',
       'data34', 'data35', 'data36', 'data37', 'data38', 'data39', 'data40',
       'data41'],
      dtype='object')


In [259]:
Y_col = cols[0]
X_col = cols[2:]

In [260]:
X, Y = data[X_col], data[[Y_col]]

In [261]:
X

,data0,data1,data2,data3,data4,data5,data6,data7,data8,data9,...,data32,data33,data34,data35,data36,data37,data38,data39,data40,data41
909,0.284153,1.000000,0.524590,0.930233,0.754098,0.573643,0.890710,0.286822,1.000000,0.131783,...,0.185792,0.759690,0.043716,0.248062,0.049180,0.217054,0.021858,0.534884,0.000000,0.697674
898,0.160870,1.000000,0.469565,0.911458,0.726087,0.552083,0.869565,0.239583,1.000000,0.015625,...,0.269565,0.395833,0.000000,0.151042,0.104348,0.093750,0.134783,0.348958,0.095652,0.385417
2845,0.197411,1.000000,0.433657,0.945238,0.647249,0.819048,0.838188,0.716667,1.000000,0.654762,...,0.262136,0.769048,0.000000,0.621429,0.042071,0.604762,0.100324,0.726190,0.126214,0.759524
167,0.567376,1.000000,0.893617,0.880102,1.000000,0.693878,0.872340,0.525510,0.673759,0.397959,...,0.460993,0.673469,0.056738,0.645408,0.000000,0.573980,0.170213,0.681122,0.234043,0.719388
1192,1.000000,1.000000,0.660256,0.728324,0.467949,0.419075,0.397436,0.190751,0.480769,0.000000,...,0.467949,0.760116,0.929487,0.849711,0.256410,0.858382,0.365385,0.887283,0.589744,0.893064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1333,0.047619,0.813665,0.303571,0.611801,0.523810,0.357143,0.589286,0.167702,0.559524,0.000000,...,0.369048,0.816770,0.000000,0.931677,0.595238,0.987578,0.476190,1.000000,0.250000,0.959627
1766,0.805310,1.000000,0.491150,0.921397,0.287611,0.768559,0.283186,0.628821,0.407080,0.513100,...,0.477876,0.733624,1.000000,0.648472,0.725664,0.644105,0.606195,0.744541,0.597345,0.814410
3369,0.058594,1.000000,0.363281,0.958656,0.648438,0.819121,0.820312,0.687339,1.000000,0.596899,...,0.199219,0.744186,0.003906,0.604651,0.000000,0.534884,0.039062,0.648579,0.074219,0.726098
2662,0.569536,1.000000,0.390728,0.865248,0.274834,0.661939,0.142384,0.484634,0.000000,0.368794,...,0.658940,0.725768,0.960265,0.699764,1.000000,0.683215,0.844371,0.777778,0.768212,0.784870


In [262]:
Y

,label
909,1
898,1
2845,5
167,0
1192,2
...,...
1333,2
1766,3
3369,6
2662,5


In [263]:
Y.value_counts()

label
2        527
0        519
4        515
5        508
3        507
1        503
6        503
Name: count, dtype: int64

In [264]:
l = len(X)
split = (0.8 * l).__floor__()
print(split)

2865


In [265]:
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

In [266]:
print(X_test)

         data0     data1     data2     data3     data4     data5     data6  \
2660  0.587248  1.000000  0.399329  0.871981  0.281879  0.669082  0.154362   
355   0.585366  1.000000  0.231707  0.880407  0.060976  0.674300  0.073171   
315   0.790514  1.000000  0.505929  0.860412  0.316206  0.643021  0.237154   
959   0.045267  1.000000  0.374486  0.986239  0.699588  0.619266  0.872428   
1044  1.000000  0.909657  0.876404  0.644860  0.632959  0.395639  0.426966   
...        ...       ...       ...       ...       ...       ...       ...   
1333  0.047619  0.813665  0.303571  0.611801  0.523810  0.357143  0.589286   
1766  0.805310  1.000000  0.491150  0.921397  0.287611  0.768559  0.283186   
3369  0.058594  1.000000  0.363281  0.958656  0.648438  0.819121  0.820312   
2662  0.569536  1.000000  0.390728  0.865248  0.274834  0.661939  0.142384   
942   0.000000  1.000000  0.356784  0.956731  0.673367  0.649038  0.834171   

         data7     data8     data9  ...    data32    data33    

In [267]:
ffnn = FFNN(128, 7, 0.5)
loss_fn = nn.CrossEntropyLoss()
ffnn = ffnn.to(device)
loss_fn = loss_fn.to(device)

X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float).to(device)
X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float).to(device)
Y_train = torch.tensor(Y_train.to_numpy(), dtype=torch.int64).reshape(-1).to(device)
Y_test = torch.tensor(Y_test.to_numpy(), dtype=torch.int64).reshape(-1).to(device)

optim = torch.optim.Adam(ffnn.parameters())

epochs = 301

for epoch in range(1, epochs):
    optim.zero_grad()

    Y_hat = ffnn(X_train)

    loss = loss_fn(Y_hat, Y_train)
    accuracy = torch.mean((torch.argmax(Y_hat, dim=1) == Y_train).float()).float()

    print(f'Epoch: {epoch}, Accuracy: {accuracy}, Loss: {loss.item()}')

    loss.backward()
    optim.step()

Y_test_hat = ffnn(X_test)
accuracy = torch.mean((torch.argmax(Y_test_hat, dim=1) == Y_test).float()).float()
print(f'Testing accuracy: {accuracy}')
torch.save(ffnn.state_dict(), '.\\model.pickle')

Epoch: 1, Accuracy: 0.13089005649089813, Loss: 1.950463891029358
Epoch: 2, Accuracy: 0.1357766091823578, Loss: 1.9473222494125366
Epoch: 3, Accuracy: 0.14624781906604767, Loss: 1.9447627067565918
Epoch: 4, Accuracy: 0.1518324613571167, Loss: 1.9434430599212646
Epoch: 5, Accuracy: 0.17102967202663422, Loss: 1.940143346786499
Epoch: 6, Accuracy: 0.19825479388237, Loss: 1.9364290237426758
Epoch: 7, Accuracy: 0.21361257135868073, Loss: 1.9340406656265259
Epoch: 8, Accuracy: 0.23350785672664642, Loss: 1.929067611694336
Epoch: 9, Accuracy: 0.27260035276412964, Loss: 1.9229764938354492
Epoch: 10, Accuracy: 0.2712042033672333, Loss: 1.9195791482925415
Epoch: 11, Accuracy: 0.2938917875289917, Loss: 1.9114437103271484
Epoch: 12, Accuracy: 0.31832462549209595, Loss: 1.9049228429794312
Epoch: 13, Accuracy: 0.320069819688797, Loss: 1.89642333984375
Epoch: 14, Accuracy: 0.31797558069229126, Loss: 1.8891289234161377
Epoch: 15, Accuracy: 0.35811519622802734, Loss: 1.8782941102981567
Epoch: 16, Accurac